In [ ]:
from google.colab import files
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
files.upload();

Saving CRNetwork.py to CRNetwork.py


In [ ]:
from CRNetwork import *
env = CRNetwork()

In [ ]:
"""
Grey Wolf Optimizer (GWO) benchmark for the CRNetwork underlay power-allocation env.

"""
import time
import numpy as np


def gwo_optimize(g_int, g_sr, pmax, n0, ith, beta,
                 pop=20, iters=30, rng=None, reward_mode="custom"):

    rng = np.random.default_rng() if rng is None else rng
    g_int = np.atleast_2d(np.asarray(g_int, dtype=np.float64))
    g_sr = np.atleast_2d(np.asarray(g_sr, dtype=np.float64))
    B, k = g_int.shape
    gi = g_int[:, None, :]
    gs = g_sr[:, None, :]

    def fitness(f):
        p = f * pmax
        cap = np.log2(1.0 + p * gs / n0).sum(-1)
        interf = (p * gi).sum(-1)
        viol = np.maximum(0.0, (interf - ith) / ith)
        if reward_mode == "paper":
            return np.where(interf <= ith, cap, -interf)
        return cap - beta * viol

    X = rng.random((B, pop, k))
    fit = fitness(X)
    idx = np.arange(B)[:, None]

    for t in range(iters):
        a = 2.0 * (1.0 - t / iters)
        order = np.argsort(-fit, axis=1)[:, :3]
        leaders = X[idx, order]
        Xn = np.zeros_like(X)
        for j in range(3):
            Xl = leaders[:, j:j + 1, :]
            r1 = rng.random((B, pop, k))
            r2 = rng.random((B, pop, k))
            A = 2.0 * a * r1 - a
            C = 2.0 * r2
            D = np.abs(C * Xl - X)
            Xn += Xl - A * D
        X = np.clip(Xn / 3.0, 0.0, 1.0)
        fit = fitness(X)

    best = np.argmax(fit, axis=1)
    return X[np.arange(B), best]


def evaluate_gwo(env, episodes, seed0, pop=20, iters=30, seed=0):

    rng = np.random.default_rng(seed)
    k = env.k
    caps, ints, viols, rews, excs, powers = [], [], [], [], [], []
    sec, calls = 0.0, 0

    for e in range(episodes):
        obs, _ = env.reset(seed=seed0 + e)
        c = i = v = r = x = 0.0
        pw = []
        for j in range(env.max_steps):

            g_int_est = np.asarray(env.hint_est, dtype=np.float64)
            g_sr = np.asarray(env.g_sr, dtype=np.float64)

            t0 = time.perf_counter()
            f = gwo_optimize(g_int_est[None, :], g_sr[None, :],
                             env.Pmax, env.N0, env.Ith, env.beta,
                             pop=pop, iters=iters, rng=rng,
                             reward_mode=env.reward_mode)[0]
            sec += time.perf_counter() - t0
            calls += 1

            action = (2.0 * f - 1.0).astype(np.float32)   # fractions -> env action
            obs, reward, term, trunc, info = env.step(action)

            c += float(np.sum(info["capacity"]))
            i += float(info["interference"])
            v += int(info["interference"] > env.Ith)
            r += float(reward)
            x += float(info["normalized_excess"])
            pw.append(info["transmit_powers"])
            if term or trunc:
                break
        n = j + 1
        caps.append(c / n); ints.append(i / n); viols.append(v / n)
        rews.append(r / n); excs.append(x / n); powers.append(np.mean(pw, axis=0))

    ints = np.asarray(ints)
    return {
        "capacity": np.asarray(caps),
        "interference_w": ints,
        "interference_dbm": 10 * np.log10(ints / 1e-3),
        "violation_rate": np.asarray(viols),
        "reward": np.asarray(rews),
        "mean_norm_excess": np.asarray(excs),
        "powers_w": np.asarray(powers),
        "ms_per_decision": 1000.0 * sec / max(calls, 1),
    }


def print_gwo_summary(res):
    print("=" * 46)
    print("                GWO BENCHMARK")
    print("=" * 46)
    print(f"Mean reward           : {res['reward'].mean():.4f}")
    print(f"Mean capacity         : {res['capacity'].mean():.4f} +/- {res['capacity'].std():.4f} bps/Hz")
    print(f"Mean interference     : {10*np.log10(res['interference_w'].mean()/1e-3):.3f} dBm")
    print(f"Violation rate        : {100*res['violation_rate'].mean():.2f}%")
    print(f"Mean normalised excess: {res['mean_norm_excess'].mean():.4f}")
    print(f"SU mean powers (mW)   : {1e3*res['powers_w'].mean(axis=0)}")
    print(f"Execution time        : {res['ms_per_decision']:.3f} ms/decision")
    print("=" * 46)


In [ ]:
res = evaluate_gwo(env, episodes = 1000, seed0 = 0, pop = 20, iters = 30, seed = 0)

In [ ]:
print_gwo_summary(res)

                GWO BENCHMARK
Mean reward           : 9.9290
Mean capacity         : 18.7337 +/- 0.6448 bps/Hz
Mean interference     : 2.348 dBm
Violation rate        : 45.35%
Mean normalised excess: 0.2333
SU mean powers (mW)   : [2.29237545 2.29336841]
Execution time        : 2.414 ms/decision


In [ ]:
OUTPUT_DIR = Path("GWO_Results")
OUTPUT_DIR.mkdir(exist_ok=True)
SMOOTH_WINDOW = 500

In [ ]:
episodes = np.arange(len(res["capacity"]))
results_df = pd.DataFrame({
    "episode": episodes,
    "capacity": res["capacity"],
    "interference": res["interference_w"],
    "violation_rate": res["violation_rate"],
    "reward": res["reward"],
    "mean_norm_excess": res["mean_norm_excess"],
    "powers_su1_w": res["powers_w"][:, 0],
    "powers_su2_w": res["powers_w"][:, 1],
    "ms_per_decision": res["ms_per_decision"],
})
results_df.to_csv(OUTPUT_DIR / "gwo_results_extended.csv", index=False)